# 03 - Applying the Chosen Model

This notebook shows how the selected configuration was trained separately for five repositories and applied to the official test set. It displays the saved executed result by default.


## Step 1 - Locate the project and load the executed final run


In [1]:
# Step 1: Locate the project root and import our package from src/.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "issue_classifier").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Could not find AI4SE_FINAL_PROJECT/src/issue_classifier")
    PROJECT_ROOT = PROJECT_ROOT.parent

source_dir = str(PROJECT_ROOT / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\AI4SE_FINAL_PROJECT


In [2]:
# Step 1: Load the selection, final metrics, manifest, and row-level predictions.
import json
import pandas as pd

selection = json.loads(
    (PROJECT_ROOT / "results" / "cross_validation.json").read_text(encoding="utf-8")
)
result_dir = PROJECT_ROOT / "results" / "chosen-model-seed42"
metrics = json.loads((result_dir / "metrics.json").read_text(encoding="utf-8"))
manifest = json.loads((result_dir / "manifest.json").read_text(encoding="utf-8"))
predictions = pd.read_csv(result_dir / "predictions.csv")

print(f"Selected configuration: {selection['selected_candidate']}")
print(f"Final run: {manifest['run_id']}")
print(f"Primary predictions: {(predictions['policy'] == 'primary').sum():,}")


Selected configuration: word_char_tfidf_svc_c1
Final run: chosen-model-seed42
Primary predictions: 1,500


## Step 2 - Show the five repository-specific models


In [3]:
# Step 2: Each repository receives a separately fitted pipeline with the same chosen parameters.
model_table = pd.DataFrame(manifest["model_files"])[
    ["repository", "filename", "sha256"]
].copy()
model_table["training_rows"] = 300
model_table["test_rows"] = 300
model_table["sha256"] = model_table["sha256"].str.slice(0, 12) + "..."
model_table


,repository,filename,sha256,training_rows,test_rows
0,bitcoin/bitcoin,bitcoin_bitcoin.joblib,506b798f7852...,300,300
1,facebook/react,facebook_react.joblib,335bc1592247...,300,300
2,microsoft/vscode,microsoft_vscode.joblib,d87e2ba31056...,300,300
3,opencv/opencv,opencv_opencv.joblib,1f4e41cb9ef2...,300,300
4,tensorflow/tensorflow,tensorflow_tensorflow.joblib,954ade821b36...,300,300


## Step 3 - Apply the matching model to each repository


In [4]:
# Step 3: Summarize the primary predictions produced by each repository model.
primary_predictions = predictions.loc[predictions["policy"] == "primary"].copy()
routed_rows = primary_predictions.groupby("repo").size().rename("test_rows").to_frame()
routed_rows["model_file"] = [
    repo.replace("/", "_") + ".joblib" for repo in routed_rows.index
]
routed_rows


,test_rows,model_file
repo,,
bitcoin/bitcoin,300,bitcoin_bitcoin.joblib
facebook/react,300,facebook_react.joblib
microsoft/vscode,300,microsoft_vscode.joblib
opencv/opencv,300,opencv_opencv.joblib
tensorflow/tensorflow,300,tensorflow_tensorflow.joblib


## Step 4 - Evaluate the official and leakage-sensitive policies


In [5]:
# Step 4: Compare the untouched 1,500-row result with the 1,497-row overlap check.
policy_table = pd.DataFrame(
    [
        {
            "policy": policy,
            "included_rows": values["included_count"],
            "excluded_rows": values["excluded_count"],
            "cross_repository_macro_f1": values["global"]["macro_f1"],
        }
        for policy, values in metrics["policies"].items()
    ]
).set_index("policy")
policy_table.style.format({"cross_repository_macro_f1": "{:.6f}"})


,included_rows,excluded_rows,cross_repository_macro_f1
policy,,,
leakage_sensitive,1497,3,0.754642
primary,1500,0,0.754385


In [6]:
# Step 4 continued: Display repository macro-F1 values from the primary evaluation.
primary = metrics["policies"]["primary"]
repository_scores = pd.Series(
    {
        repo: values["macro"]["f1"]
        for repo, values in primary["repositories"].items()
    },
    name="macro_f1",
).sort_values(ascending=False)
repository_scores.to_frame().style.format("{:.6f}")


,macro_f1
facebook/react,0.807182
opencv/opencv,0.783185
tensorflow/tensorflow,0.753815
microsoft/vscode,0.727126
bitcoin/bitcoin,0.700619


## Step 5 - Inspect individual mistakes


In [7]:
# Step 5: Show concrete misclassified rows for later error analysis.
mistakes = primary_predictions.loc[
    primary_predictions["true_label"] != primary_predictions["predicted_label"],
    ["row_index", "repo", "true_label", "predicted_label", "is_overlap"],
]
mistakes.head(12)


,row_index,repo,true_label,predicted_label,is_overlap
9,909,bitcoin/bitcoin,feature,question,False
28,928,bitcoin/bitcoin,feature,bug,False
31,931,bitcoin/bitcoin,feature,question,False
42,942,bitcoin/bitcoin,feature,question,False
63,963,bitcoin/bitcoin,feature,bug,True
65,965,bitcoin/bitcoin,feature,question,False
70,970,bitcoin/bitcoin,feature,question,False
79,979,bitcoin/bitcoin,feature,bug,False
88,988,bitcoin/bitcoin,feature,bug,False
106,1006,bitcoin/bitcoin,question,feature,False


## Optional - Repeat final training and evaluation


In [8]:
# Optional step: Refit all five pipelines and overwrite the saved final result only when requested.
RERUN_FINAL_EVALUATION = False

if RERUN_FINAL_EVALUATION:
    from issue_classifier import CsvIssueStore, run_final_chosen_evaluation

    store = CsvIssueStore(PROJECT_ROOT / "official_nlbse24" / "data")
    rerun = run_final_chosen_evaluation(
        store,
        selection_path=PROJECT_ROOT / "results" / "cross_validation.json",
        output_root=PROJECT_ROOT / "results" / "chosen-model-seed42",
        model_root=PROJECT_ROOT / "artifacts" / "chosen-model-seed42" / "models",
    )
    print(rerun["metrics"]["policies"]["primary"]["global"]["macro_f1"])
else:
    print("Using the saved executed final evaluation.")


Using the saved executed final evaluation.
